In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('./dataset/MIC.csv')


In [3]:
# preprocessing the data
# Same thing performed in the previous notebook, just lowercasing and removing additional whitespaces
# work on both of the questions and the chatbot answer. if it helps, create a new column with a combined text of the question and answer, and then work on that column. 
train_texts_raw = df.loc[df['split'] == 'train', 'QA'].drop_duplicates().tolist()
test_texts_raw  = df.loc[df['split'] == 'test',  'QA'].drop_duplicates().tolist()

print(f"Unique train QA sequences: {len(train_texts_raw)}")
print(f"Unique test QA sequences:  {len(test_texts_raw)}")


Unique train QA sequences: 30248
Unique test QA sequences:  3781


In [4]:
import re

def simple_preprocess(text: str) -> str:
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [5]:
TOKEN_PATTERN = re.compile(r"[a-z0-9]+(?:'[a-z]+)?|[.,!?;:]")

def tokenize(text: str):
    return TOKEN_PATTERN.findall(text)

In [6]:
# Preprocess and tokenize the train and test texts
train_sentences = [tokenize(simple_preprocess(text)) for text in train_texts_raw]
test_sentences  = [tokenize(simple_preprocess(text)) for text in test_texts_raw]

A vocabulary size definition is then processed. We define a minimum word count number that we use as the mimimum frequency a token needs to have in the corpus to not get replaced by the literal `<UNK>`. We do this because later when doing smoothing and counting:

- We need to fix our Vocabulary size. We can't just add k for the smoothing to normalize the probability distribution for tokens the model didn't see before.

- `<UNK>` then becomes its own token that represents a rare word in the corpus. If there are multiple rare words, there will be more number of `<UNK>` in the corpus. The probability of the UNK represents the likelihood of a rare word appearing.

In [7]:
from collections import Counter, defaultdict

UNK = "<UNK>"
MIN_COUNT = 2

word_counts = Counter(w for sent in train_sentences for w in sent)
vocab = {w for w, c in word_counts.items() if c >= MIN_COUNT}
print(f"Raw vocabulary size: {len(word_counts)}")
print(f"Vocabulary size after MIN_COUNT={MIN_COUNT} cutoff: {len(vocab)}")

Raw vocabulary size: 23086
Vocabulary size after MIN_COUNT=2 cutoff: 14871


In [8]:
def apply_unk(sentence):
    return [w if w in vocab else UNK for w in sentence]


train_sentences = [apply_unk(s) for s in train_sentences]
test_sentences  = [apply_unk(s) for s in test_sentences]

Pad sentences should also be added for each QA. Up to n-1 `<s>` pads and only one `</s>` pad. This is to show the start and end of sentences and differentiate among unrelated tokens.

In [9]:
BOS, EOS = "<s>", "</s>"

def pad_sentence(sentence, n):
    """Pad with n-1 <s> tokens at the start (so an n-gram model always has full
    context at the start of a sequence) and a single </s> at the end."""
    return [BOS] * (n - 1) + sentence + [EOS]

In [10]:
# Unigram Model
unigram_sents = [pad_sentence(s, 1) for s in train_sentences]
unigram_counts = Counter(w for sent in unigram_sents for w in sent)
total_unigrams = sum(unigram_counts.values())
V = len(unigram_counts)  # vocabulary size, including <UNK> and </s>

print(f"Unigram vocabulary size (incl. <UNK>/</s>): {V}")
print(f"Total unigram tokens: {total_unigrams}")
print("Top 10 unigrams:", unigram_counts.most_common(10))


Unigram vocabulary size (incl. <UNK>/</s>): 14873
Total unigram tokens: 1665466
Top 10 unigrams: [('a', 71304), ('.', 69092), (':', 61501), ('the', 51228), ('i', 50342), ('to', 45939), (',', 45204), ('?', 36289), ('you', 33398), ('q', 30256)]


In [11]:
# bigram model
bigram_sents = [pad_sentence(s, 2) for s in train_sentences]

bigram_counts = defaultdict(Counter)   # bigram_counts[w1][w2] -> count
bigram_context_counts = Counter()      # bigram_context_counts[w1] -> count of w1 as a bigram context

for sent in bigram_sents:
    for w1, w2 in zip(sent[:-1], sent[1:]):
        bigram_counts[w1][w2] += 1
        bigram_context_counts[w1] += 1

print(f"Distinct bigram contexts: {len(bigram_counts)}")
print(f"Total bigrams seen: {sum(bigram_context_counts.values())}")
print("Most common continuations of 'i':", bigram_counts['i'].most_common(5))


Distinct bigram contexts: 14873
Total bigrams seen: 1665466
Most common continuations of 'i': [("don't", 9575), ('think', 6737), ('would', 4947), ('was', 3097), ('have', 2597)]


In [12]:
# trigram model
trigram_sents = [pad_sentence(s, 3) for s in train_sentences]

trigram_counts = defaultdict(Counter)   # trigram_counts[(w1, w2)][w3] -> count
trigram_context_counts = Counter()      # trigram_context_counts[(w1, w2)] -> count of (w1, w2) as a trigram context

for sent in trigram_sents:
    for w1, w2, w3 in zip(sent[:-2], sent[1:-1], sent[2:]):
        trigram_counts[(w1, w2)][w3] += 1
        trigram_context_counts[(w1, w2)] += 1

print(f"Distinct trigram contexts: {len(trigram_counts)}")
print(f"Total trigrams seen: {sum(trigram_context_counts.values())}")
print("Most common continuations of ('<s>', '<s>'):", trigram_counts[('<s>', '<s>')].most_common(5))


Distinct trigram contexts: 263667
Total trigrams seen: 1665466
Most common continuations of ('<s>', '<s>'): [('q', 30248)]


Saving the models

In [13]:
import os
import pickle

MODELS_DIR = "./models"
os.makedirs(MODELS_DIR, exist_ok=True)

unigram_model = {
    "n": 1,
    "counts": dict(unigram_counts),
    "total": total_unigrams,
}

bigram_model = {
    "n": 2,
    "counts": {w1: dict(c2) for w1, c2 in bigram_counts.items()},
    "context_counts": dict(bigram_context_counts),
}

trigram_model = {
    "n": 3,
    "counts": {ctx: dict(c3) for ctx, c3 in trigram_counts.items()},
    "context_counts": dict(trigram_context_counts),
}

for name, model in [("unigram_model", unigram_model),
                     ("bigram_model", bigram_model),
                     ("trigram_model", trigram_model)]:
    with open(os.path.join(MODELS_DIR, f"{name}.pkl"), "wb") as f:
        pickle.dump(model, f)

# persist the vocabulary too, so a reloaded model applies the same <UNK> mapping
with open(os.path.join(MODELS_DIR, "vocab.pkl"), "wb") as f:
    pickle.dump(vocab, f)

print(f"Saved unigram_model.pkl, bigram_model.pkl, trigram_model.pkl, vocab.pkl to {MODELS_DIR}/")


Saved unigram_model.pkl, bigram_model.pkl, trigram_model.pkl, vocab.pkl to ./models/


In [14]:
# n-gram probabilities
def unigram_prob(w):
    return unigram_counts.get(w, 0) / total_unigrams


def bigram_prob(w1, w2):
    context_count = bigram_context_counts.get(w1, 0)
    if context_count == 0:
        return 0.0
    return bigram_counts[w1].get(w2, 0) / context_count


def trigram_prob(w1, w2, w3):
    context_count = trigram_context_counts.get((w1, w2), 0)
    if context_count == 0:
        return 0.0
    return trigram_counts[(w1, w2)].get(w3, 0) / context_count


In [15]:
# probability with laplace smoothing (Add k)
def unigram_prob_addk(w, k=1.0):
    return (unigram_counts.get(w, 0) + k) / (total_unigrams + k * V)


def bigram_prob_addk(w1, w2, k=1.0):
    context_count = bigram_context_counts.get(w1, 0)
    count = bigram_counts.get(w1, {}).get(w2, 0)
    return (count + k) / (context_count + k * V)


def trigram_prob_addk(w1, w2, w3, k=1.0):
    context_count = trigram_context_counts.get((w1, w2), 0)
    count = trigram_counts.get((w1, w2), {}).get(w3, 0)
    return (count + k) / (context_count + k * V)



In [ ]:
# Interpolation of unigram, bigram and trigram probabilities
LAMBDA1, LAMBDA2, LAMBDA3 = 0.1, 0.3, 0.6  # unigram, bigram, trigram weights -- must sum to 1
# lambdas can be tuned by the dev split instead of fixing/guessing them.

def trigram_prob_interp(w1, w2, w3, lambdas=(LAMBDA1, LAMBDA2, LAMBDA3)):
    l1, l2, l3 = lambdas
    p1 = unigram_prob(w3)
    p2 = bigram_prob(w2, w3)
    p3 = trigram_prob(w1, w2, w3)
    return l1 * p1 + l2 * p2 + l3 * p3


In [17]:
# perplexity of models on test set
import math

def sentence_log_prob(tokens, prob_fn, n):
    """tokens: already-<UNK>-mapped sentence, unpadded. Returns (sum of log P, count of predicted tokens)."""
    padded = pad_sentence(tokens, n)
    log_prob = 0.0
    for i in range(n - 1, len(padded)):
        if n == 1:
            p = prob_fn(padded[i])
        elif n == 2:
            p = prob_fn(padded[i - 1], padded[i])
        else:
            p = prob_fn(padded[i - 2], padded[i - 1], padded[i])
        log_prob += math.log(p)
    return log_prob, len(padded) - (n - 1)


def corpus_perplexity(sentences, prob_fn, n):
    total_log_prob = 0.0
    total_tokens = 0
    for tokens in sentences:
        lp, count = sentence_log_prob(tokens, prob_fn, n)
        total_log_prob += lp
        total_tokens += count
    return math.exp(-total_log_prob / total_tokens)


Comparing the models

In [18]:
from functools import partial

k = 1.0
pp_uni = corpus_perplexity(test_sentences, partial(unigram_prob_addk, k=k), n=1)
pp_bi  = corpus_perplexity(test_sentences, partial(bigram_prob_addk, k=k), n=2)
pp_tri = corpus_perplexity(test_sentences, partial(trigram_prob_addk, k=k), n=3)
print(f"Add-k (k={k}) perplexity  ->  unigram: {pp_uni:.2f}  bigram: {pp_bi:.2f}  trigram: {pp_tri:.2f}")

pp_interp = corpus_perplexity(test_sentences, trigram_prob_interp, n=3)
print(f"Interpolated trigram perplexity: {pp_interp:.2f}")


Add-k (k=1.0) perplexity  ->  unigram: 368.38  bigram: 270.28  trigram: 1350.60
Interpolated trigram perplexity: 61.46


In [19]:
try:
    pp_mle_tri = corpus_perplexity(test_sentences, trigram_prob, n=3)
    print(pp_mle_tri)
except ValueError as e:
    print(f"Raw MLE trigram perplexity is undefined: {e}")


Raw MLE trigram perplexity is undefined: math domain error


In [20]:
# Smoothing/Interpolation changing predictions
import random

def predict_next_mle(w1, w2):
    """Top next-word under raw MLE trigram. Returns (None, 0.0) if the context
    was never seen in training -- MLE literally has no basis for a prediction."""
    candidates = trigram_counts.get((w1, w2))
    if not candidates:
        return None, 0.0
    word, count = candidates.most_common(1)[0]
    context_count = trigram_context_counts[(w1, w2)]
    return word, count / context_count


def predict_next_interp(w1, w2, candidate_words):
    best_word, best_prob = None, -1.0
    for w in candidate_words:
        p = trigram_prob_interp(w1, w2, w)
        if p > best_prob:
            best_word, best_prob = w, p
    return best_word, best_prob


candidate_words = vocab | {"</s>"}

# distinct (w1, w2) trigram contexts actually occurring in the test set
test_contexts = set()
for tokens in test_sentences:
    padded = pad_sentence(tokens, 3)
    for i in range(2, len(padded)):
        test_contexts.add((padded[i - 2], padded[i - 1]))

random.seed(7)
test_contexts = list(test_contexts)
random.shuffle(test_contexts)

examples = []
for w1, w2 in test_contexts:
    mle_word, mle_prob = predict_next_mle(w1, w2)
    interp_word, interp_prob = predict_next_interp(w1, w2, candidate_words)
    if mle_word != interp_word:
        examples.append((w1, w2, mle_word, mle_prob, interp_word, interp_prob))
    if len(examples) == 5:
        break

for w1, w2, mle_word, mle_prob, interp_word, interp_prob in examples:
    seen_count = trigram_context_counts.get((w1, w2), 0)
    print(f"Context: ('{w1}', '{w2}')  [seen {seen_count} time(s) in training]")
    print(f"  MLE trigram prediction:          {mle_word!r}  (P={mle_prob:.4f})")
    print(f"  Interpolated trigram prediction: {interp_word!r}  (P={interp_prob:.4f})")
    # breakdown of what fed the interpolation winner, for your "why" explanation
    p1 = unigram_prob(interp_word)
    p2 = bigram_prob(w2, interp_word)
    p3 = trigram_prob(w1, w2, interp_word)
    print(f"  interp winner breakdown -> unigram={p1:.5f}  bigram={p2:.5f}  trigram={p3:.5f}")
    print()


Context: ('vomit', ',')  [seen 0 time(s) in training]
  MLE trigram prediction:          None  (P=0.0000)
  Interpolated trigram prediction: 'but'  (P=0.0441)
  interp winner breakdown -> unigram=0.00526  bigram=0.14510  trigram=0.00000

Context: ('risks', 'and')  [seen 0 time(s) in training]
  MLE trigram prediction:          None  (P=0.0000)
  Interpolated trigram prediction: 'i'  (P=0.0238)
  interp winner breakdown -> unigram=0.03023  bigram=0.06916  trigram=0.00000

Context: ('this', 'philosophy')  [seen 0 time(s) in training]
  MLE trigram prediction:          None  (P=0.0000)
  Interpolated trigram prediction: ','  (P=0.0573)
  interp winner breakdown -> unigram=0.02714  bigram=0.18182  trigram=0.00000

Context: ('family', 'respond')  [seen 0 time(s) in training]
  MLE trigram prediction:          None  (P=0.0000)
  Interpolated trigram prediction: 'to'  (P=0.1528)
  interp winner breakdown -> unigram=0.02758  bigram=0.50000  trigram=0.00000

Context: ('friend', 'associate')  [s